# Llama 3.2 Finetune using LoRA

In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

/Users/max/Repos/KTH/llama3-peft/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
dataset_name = "mlabonne/FineTome-100k"
output_dir = "outputs"
lora_output_dir = "lora_model"

In [3]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [4]:
max_seq_length = 1024
batch_size = 2
grad_accum = 4
lora_r = 16

In [6]:
load_in_4bit = device == "cuda"
if load_in_4bit:
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16, token=hf_token)
    model = prepare_model_for_kbit_training(model)
else:
    dtype = torch.float16 if device == "mps" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        dtype=dtype, 
        device_map=None, 
        low_cpu_mem_usage=True,
    )
    model = model.to(device)
    if device == "mps":
        torch.mps.empty_cache()

Loading checkpoint shards: 100%|██████████| 2/2 [00:13<00:00,  6.67s/it]


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
model = get_peft_model(model, LoraConfig(r=lora_r, lora_alpha=lora_r, target_modules=target_modules, lora_dropout=0, bias="none", task_type="CAUSAL_LM"))

In [9]:
dataset = load_dataset(dataset_name, split="train")

In [10]:
def standardize(example):
    standardized = []
    for msg in example.get("conversations", []):
        role = msg.get("from", "").lower()
        content = msg.get("value", "")
        if role == "system":
            standardized.append({"role": "system", "content": content})
        elif role in ["human", "user"]:
            standardized.append({"role": "user", "content": content})
        elif role in ["gpt", "assistant"]:
            standardized.append({"role": "assistant", "content": content})
    return {"conversations": standardized}

In [11]:
dataset = load_dataset(dataset_name, split="train")
dataset = dataset.map(standardize).map(lambda x: {"messages": x["conversations"]}, remove_columns=[c for c in dataset.column_names if c != "conversations"])

Map: 100%|██████████| 100000/100000 [00:03<00:00, 27813.53 examples/s]


In [12]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=device == "cuda" and not torch.cuda.is_bf16_supported(),
        bf16=device == "cuda" and torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit" if device == "cuda" else "adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        save_strategy="steps",
        save_steps=50,
        max_length=max_seq_length,
        packing=False,
        gradient_checkpointing=False,
        dataloader_pin_memory=False,
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
)

Truncating train dataset: 100%|██████████| 100000/100000 [00:00<00:00, 137838.58 examples/s]


In [ ]:
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)